# MemoRizz Memory Types, Zero to Hero

## Units, operations, lifecycle ownership, and selection rules

This notebook is a field guide to every current `MemoryType`. It goes
below the agent-level tour in `agent_memory.ipynb` and examines the
shape and supported operations of each store.

The most important rule is that **not every memory store should be
mutated directly**. Conversation, summaries, cache, tool logs, and
workflow evidence are normally runtime-owned because their metadata
must remain consistent with tenant, trace, and policy state.

**Use this notebook as both a lesson and a design reference.** Each
section answers six questions: what problem the type solves, what one
unit contains, who may write it, how it is retrieved, how it changes or
expires, and when another type is a better fit. The code performs real
operations and the surrounding text explains what the output proves.

By the end you should be able to design a memory schema from application
invariants instead of reaching for one undifferentiated vector store.

## Taxonomy

```mermaid
mindmap
  root((Agent memory))
    Episodic
      Conversation
      Summaries
    Semantic
      Persona
      Entity
      Knowledge base
    Procedural
      Toolbox
      Workflow
      Skillbox
      Tool log
    Working
      Short-term context
      Semantic cache
    Social
      Shared memory
    Durable definition
      MemAgent
```

The taxonomy is based on **function**, not storage technology. Several
types may use embeddings, but their authority and lifecycle differ. A
knowledge chunk is evidence, a skill is instruction, a cached answer is
a disposable optimization, and a workflow is observed procedure. Their
vectors do not make them interchangeable.

Memory can also move between layers: repeated episodes can support a
semantic claim; verified workflow traces can propose a skill; a long
episode can be compacted into a summary. These transitions require
provenance and policy because the derived unit may carry more authority
than the raw experience.

## Operation model

| Ownership | Meaning | Examples |
|---|---|---|
| Application-authored | Your code deliberately creates and maintains the unit | persona, entity, knowledge, authored skill |
| Runtime-owned | `MemAgent` writes it as part of a turn or lifecycle transition | conversation, summary, tool log, cache, workflow evidence |
| Coordination-owned | A workflow/orchestrator writes typed messages | shared memory |
| Provider-administered | Direct provider CRUD is appropriate for inspection, migration, retention, and cleanup | all stores, with exact scope |

Provider CRUD is a substrate. Prefer the high-level owner because it
preserves invariants such as provenance, source links, canonical
hashes, cache fingerprints, and audit identity.

| Lifecycle action | Examples |
|---|---|
| Create | ingest a document, append a turn, upsert an entity |
| Retrieve | exact ID, scoped history, semantic/hybrid search |
| Refine | version a persona, supersede a fact, compact a thread |
| Govern | approve a side effect, constrain tenant scope, audit a tool result |
| Forget | TTL expiry, cache invalidation, skill demotion, scoped deletion |

**Safety rule:** direct provider access is powerful and is therefore an
administrative boundary. Application code should not manufacture
runtime-owned rows that pretend a tool ran, a workflow succeeded, or a
human approved an action.

## 0 · Local, reproducible setup

This notebook uses an isolated filesystem provider, deterministic
embeddings, and a deterministic teaching model. It does not use an API
key and does not make network requests.

The local components are deliberately simple so the output is stable.
They validate shapes, scope, lifecycle methods, and invariants; they do
not estimate production retrieval quality. The Oracle companion covers
a real database provider, while the evaluation suite is the correct
place to compare recall, grounding, latency, and cost.

In [1]:
import hashlib
import math
import tempfile
from pathlib import Path
from types import SimpleNamespace

from memorizz import (
    EntityMemory,
    FileSystemConfig,
    FileSystemProvider,
    KnowledgeBase,
    MemAgent,
    MemAgentBuilder,
    MemoryType,
    Persona,
    RoleType,
    SharedMemory,
    Toolbox,
    ToolResultPolicy,
    governed_tool,
)
from memorizz.embeddings import set_global_embedding_manager
from memorizz.long_term.procedural.skillbox import Skill, SkillStatus, Skillbox
from memorizz.long_term.procedural.workflow import Workflow, WorkflowOutcome


class TutorialEmbeddings:
    dimensions = 64

    def get_embedding(self, text: str, **_kwargs):
        vector = [0.0] * self.dimensions
        for token in str(text).lower().split():
            digest = hashlib.sha256(token.encode("utf-8")).digest()
            vector[int.from_bytes(digest[:2], "big") % self.dimensions] += 1.0
        norm = math.sqrt(sum(value * value for value in vector)) or 1.0
        return [value / norm for value in vector]

    def get_embeddings(self, texts, **kwargs):
        return [self.get_embedding(text, **kwargs) for text in texts]

    def get_dimensions(self):
        return self.dimensions

    def get_default_model(self):
        return "tutorial-hash-v1"

    def get_provider_info(self):
        return {
            "provider": "tutorial",
            "model": self.get_default_model(),
            "dimensions": self.dimensions,
            "config": {},
        }


class TutorialModel:
    model = "tutorial-memory-types"

    def __init__(self):
        self.calls = 0
        self._usage = {}

    def generate(self, messages, tools=None):
        self.calls += 1
        text = "\n".join(
            str(message.get("content") or "")
            for message in messages
            if isinstance(message, dict)
        )
        self._usage = {
            "prompt_tokens": max(1, len(text) // 4),
            "completion_tokens": 12,
            "total_tokens": max(1, len(text) // 4) + 12,
        }
        lowered = text.lower()
        if "comprehensive but concise summary" in lowered:
            return "The user prefers concise Python and is building an Oracle-backed API."
        tool_messages = [m for m in messages if isinstance(m, dict) and m.get("role") == "tool"]
        if tool_messages:
            return "The service is healthy."
        users = [
            str(m.get("content") or "")
            for m in messages
            if isinstance(m, dict) and m.get("role") == "user"
        ]
        latest = users[-1].lower() if users else ""
        if tools and "service health" in latest:
            call = SimpleNamespace(
                id=f"memory-types-call-{self.calls}",
                function=SimpleNamespace(name="service_health", arguments='{"service":"retrieval-api"}'),
            )
            return SimpleNamespace(
                choices=[SimpleNamespace(message=SimpleNamespace(content=None, tool_calls=[call]))]
            )
        if "what do i prefer" in latest and "concise python" in lowered:
            return "You prefer concise Python examples."
        return "Recorded."

    def get_config(self):
        return {"provider": "tutorial", "model": self.model}

    def get_last_usage(self):
        return dict(self._usage)


embeddings = TutorialEmbeddings()
set_global_embedding_manager(embeddings)
ROOT = Path(tempfile.mkdtemp(prefix="memorizz-memory-types-"))
provider = FileSystemProvider(
    FileSystemConfig(root_path=ROOT, use_faiss=False, embedding_provider=embeddings)
)

MEMORY_ID = "memory-types-demo"
USER_ID = "developer-42"
THREAD_ID = "memory-taxonomy"
SCOPE = {"memory_id": MEMORY_ID, "user_id": USER_ID, "thread_id": THREAD_ID}
print(
    {
        "backend": "isolated temporary filesystem",
        "types": [item.value for item in MemoryType],
    }
)

{'backend': 'isolated temporary filesystem', 'types': ['personas', 'toolbox', 'entity_memory', 'short_term_memory', 'knowledge_base', 'conversation_memory', 'workflow_memory', 'skillbox', 'agents', 'shared_memory', 'summaries', 'semantic_cache', 'tool_log']}


## 1 · The provider contract

Every backend exposes a common persistence surface: `store`,
`store_many`, `retrieve_by_id`, `retrieve_by_query`, `list_all`,
`update_by_id`, and `delete_by_id`. This is useful for administration
and custom memory units, but high-level components should own normal
application writes.

The CRUD cell intentionally creates, reads, updates, and deletes one
disposable knowledge row. Its output proves the mechanical provider
contract. It does **not** prove that the row has complete ingestion
provenance, chunk metadata, or agent attachment; `KnowledgeBase` owns
those higher-level invariants.

Ask every provider for `memory_capabilities()` before depending on batch
atomicity, native vector search, hybrid search, scores, or provenance.
A shared method name does not imply equal operational characteristics.

In [2]:
low_level_id = provider.store(
    {
        "content": "A low-level provider contract example.",
        "namespace": "provider-demo",
        "memory_id": MEMORY_ID,
        "user_id": USER_ID,
        "embedding": embeddings.get_embedding("provider contract example"),
    },
    MemoryType.KNOWLEDGE_BASE,
)
print("CREATE:", low_level_id)
print("READ  :", provider.retrieve_by_id(low_level_id, MemoryType.KNOWLEDGE_BASE)["content"])
provider.update_by_id(
    low_level_id,
    {"content": "An updated low-level provider contract example."},
    MemoryType.KNOWLEDGE_BASE,
)
print("UPDATE:", provider.retrieve_by_id(low_level_id, MemoryType.KNOWLEDGE_BASE)["content"])
print("DELETE:", provider.delete_by_id(low_level_id, MemoryType.KNOWLEDGE_BASE))

CREATE: c10b8414-1790-4741-a433-99f53fc10100
READ  : A low-level provider contract example.
UPDATE: An updated low-level provider contract example.
DELETE: True


---
# Part I · Episodic memory

## 2 · `CONVERSATION_MEMORY`

**Unit shape:** role, content, timestamp, memory/thread/user/agent IDs,
optional embedding, provenance, and optional `summary_id`.

**Owner:** `MemAgent.run()` or `run_stream()`.

**Operations:** write through the agent; read through scoped history;
update/delete only for administrative retention or correction.

**Why it exists:** chronology matters for follow-up questions,
commitments, decisions, and debugging. The runtime writes both sides of
the exchange after assembling the turn context, preserving which agent,
user, memory space, and thread produced the event.

**Read the output:** the first run forms the preference episode; the
second retrieves it. The printed rows expose the role and three scope
identifiers, which are stronger evidence than a plausible response.

**Use instead:** move stable, independently useful facts to entity
memory; use summaries when the episode is too large to replay. Apply a
retention policy when raw conversational detail is no longer needed or
must be deleted for privacy.

In [3]:
episodic_agent = (
    MemAgentBuilder()
    .with_name("Memory Types Guide")
    .with_instruction("Remember scoped user preferences.")
    .with_model(TutorialModel())
    .with_memory_provider(provider)
    .with_memory_ids(MEMORY_ID)
    .with_automations_enabled(False)
    .build_and_save()
)
episodic_agent.run("I prefer concise Python examples.", **SCOPE)
print(episodic_agent.run("What do I prefer?", **SCOPE))

conversation = provider.retrieve_conversation_history_ordered_by_timestamp(
    memory_id=MEMORY_ID,
    memory_type=MemoryType.CONVERSATION_MEMORY,
    user_id=USER_ID,
    thread_id=THREAD_ID,
)
print(
    [
        {key: row.get(key) for key in ("_id", "role", "content", "memory_id", "user_id", "thread_id")}
        for row in conversation
    ]
)

You prefer concise Python examples.
[{'_id': '19969678-de5f-4229-a12f-830fb6b237e4', 'role': 'user', 'content': 'I prefer concise Python examples.', 'memory_id': 'memory-types-demo', 'user_id': 'developer-42', 'thread_id': 'memory-taxonomy'}, {'_id': '1504c2dd-f298-4227-af53-26bb785cf08a', 'role': 'assistant', 'content': 'Recorded.', 'memory_id': 'memory-types-demo', 'user_id': 'developer-42', 'thread_id': 'memory-taxonomy'}, {'_id': 'cbf7cdec-c246-41bc-b538-ae18256930af', 'role': 'user', 'content': 'What do I prefer?', 'memory_id': 'memory-types-demo', 'user_id': 'developer-42', 'thread_id': 'memory-taxonomy'}, {'_id': '964a25a9-89dd-484e-a13b-38d2c6c62c4c', 'role': 'assistant', 'content': 'You prefer concise Python examples.', 'memory_id': 'memory-types-demo', 'user_id': 'developer-42', 'thread_id': 'memory-taxonomy'}]


## 3 · `SUMMARIES`

**Unit shape:** compressed content, `source_message_ids`,
`period_start`, `period_end`, `memory_units_count`, and exact scope.

**Owner:** `generate_summaries(...)` and the provider's atomic
summary-link operation. Summarization marks source rows; it does not
delete them.

**Why it exists:** repeatedly injecting an entire transcript increases
prompt tokens, latency, and cost. A summary is a compact episodic index
that preserves a route back to the original evidence.

**Read the output:** `memory_units_count` must match the number of
`source_message_ids`; the period bounds show what interval was covered.
`fetch_context_summary` performs a scope-aware lookup by summary ID.

**Failure modes:** orphan summaries, source rows marked in a separate
non-atomic transaction, cross-thread compaction, and prose that omits a
critical commitment. Evaluate compression fidelity separately from the
structural assertion shown here. Deletion and summarization are
different operations.

In [4]:
summary_ids = episodic_agent.generate_summaries(
    memory_id=MEMORY_ID,
    user_id=USER_ID,
    thread_id=THREAD_ID,
    days_back=7,
    max_memories_per_summary=20,
)
summary = episodic_agent.fetch_context_summary(
    summary_ids[0],
    memory_id=MEMORY_ID,
    user_id=USER_ID,
    thread_id=THREAD_ID,
)
print(
    {
        key: summary[key]
        for key in (
            "content",
            "source_message_ids",
            "period_start",
            "period_end",
            "memory_units_count",
        )
    }
)
assert summary["memory_units_count"] == len(summary["source_message_ids"])

{'content': 'The user prefers concise Python and is building an Oracle-backed API.', 'source_message_ids': ['19969678-de5f-4229-a12f-830fb6b237e4', '1504c2dd-f298-4227-af53-26bb785cf08a', 'cbf7cdec-c246-41bc-b538-ae18256930af', '964a25a9-89dd-484e-a13b-38d2c6c62c4c'], 'period_start': 1787516360.426657, 'period_end': 1787516360.430193, 'memory_units_count': 4}


---
# Part II · Semantic memory

## 4 · `PERSONAS`

**Unit shape:** stable identity fields, embedding, version, and
`evolution_history` entries with a change trigger.

**Operations:** `store_persona`, `retrieve_persona`, `list_personas`,
`get_most_similar_persona`, `update`, and `delete_persona`.

**Why it exists:** a persona gives the agent a durable role, goals, and
behavioral frame. It is part of the agent's identity rather than a fact
about the user or world.

**Read the output:** the stored name proves retrieval, while the version
and `evolution_history` prove that the goal changed with a recorded
trigger. The disposable persona demonstrates explicit deletion.

**Governance:** do not let ordinary retrieved text silently rewrite the
persona. Changes can alter instruction behavior, so record who or what
authorized them and retain the prior version. Use entity memory for
mutable facts about people and systems.

In [5]:
persona = Persona(
    name="Platform Guide",
    role=RoleType.TECHNICAL_EXPERT,
    goals="Teach memory systems through precise examples.",
    background="An AI platform engineer.",
)
persona_storage_id = persona.store_persona(provider)
print("Stored:", Persona.retrieve_persona(persona_storage_id, provider)["name"])

update_result = persona.update(
    updates={"goals": "Teach governed memory systems through precise examples."},
    change_trigger={
        "reason": "Governance became a course requirement.",
        "source_type": "user_feedback",
        "source_id": "curriculum-v2",
    },
    provider=provider,
)
print({"version": update_result["version"], "history": persona.evolution_history})
print("List:", [row["name"] for row in Persona.list_personas(provider)])

disposable = Persona("Disposable example")
disposable_id = disposable.store_persona(provider)
print("Delete temporary persona:", Persona.delete_persona(disposable_id, provider))

Stored: Platform Guide
{'version': 2, 'history': [{'version': 2, 'timestamp': '2026-08-23T22:19:20.438461', 'changes': {'goals': {'old': 'Provide expert technical advice and troubleshoot complex problems. Teach memory systems through precise examples.', 'new': 'Teach governed memory systems through precise examples.'}}, 'change_trigger': {'reason': 'Governance became a course requirement.', 'source_type': 'user_feedback', 'source_id': 'curriculum-v2', 'conversation_id': None, 'agent_id': None, 'triggered_at': '2026-08-23T22:19:20.438434'}}]}
List: ['Platform Guide']
Delete temporary persona: True


## 5 · `ENTITY_MEMORY`

**Unit shape:** stable entity ID, type, attributes with confidence and
source, relations, metadata, embedding, and exact tenant/memory scope.

**Operations:** `upsert_entity`, `record_attribute`, `get_entity`,
`get_entity_by_name`, `list_entities`, and diagnostic semantic search.

**Why it exists:** entities provide addressable, structured state. An
application can ask for the current owner or SLO without searching every
conversation where the service was mentioned.

**Read the output:** the record contains independently sourced
attributes and a relation; diagnostics show how search was performed.
Exact scoped lookup remains the preferred path when the entity ID is
known, while semantic search helps discover an unknown ID.

**Conflict and forgetting:** treat attributes as claims with timestamp,
source, confidence, and supersession policy. An architecture record may
outrank an older chat message. Remove or supersede stale values rather
than returning all contradictory values as if they were equally current.

In [6]:
entity_memory = EntityMemory(provider)
entity_memory.upsert_entity(
    entity_id="service-retrieval-api",
    name="retrieval-api",
    entity_type="service",
    attributes=[
        {"name": "owner", "value": "Ada", "confidence": 0.98, "source": "service-catalog"},
        {"name": "database", "value": "Oracle AI Database", "confidence": 0.95, "source": "architecture-record"},
    ],
    relations=[
        {"entity_id": "index-main", "relation_type": "uses_index", "confidence": 0.9}
    ],
    memory_id=MEMORY_ID,
    user_id=USER_ID,
)
entity_memory.record_attribute(
    entity_id="service-retrieval-api",
    attribute_name="slo",
    attribute_value="p95 under 250ms",
    source="slo-catalog",
    memory_id=MEMORY_ID,
    user_id=USER_ID,
)
record = entity_memory.get_entity(
    "service-retrieval-api", memory_id=MEMORY_ID, user_id=USER_ID
)
diagnostics = entity_memory.search_entities_with_diagnostics(
    "Oracle retrieval service",
    memory_id=MEMORY_ID,
    user_id=USER_ID,
    limit=3,
)
print({"record": record, "search": diagnostics})

{'record': {'entity_id': 'service-retrieval-api', 'name': 'retrieval-api', 'entity_type': 'service', 'memory_id': 'memory-types-demo', 'user_id': 'developer-42', 'attributes': [{'name': 'owner', 'value': 'Ada', 'confidence': 0.98, 'source': 'service-catalog', 'created_at': '2026-08-23T21:19:20.442350', 'updated_at': '2026-08-23T21:19:20.442350'}, {'name': 'database', 'value': 'Oracle AI Database', 'confidence': 0.95, 'source': 'architecture-record', 'created_at': '2026-08-23T21:19:20.442350', 'updated_at': '2026-08-23T21:19:20.442350'}, {'name': 'slo', 'value': 'p95 under 250ms', 'confidence': 0.85, 'source': 'slo-catalog', 'created_at': '2026-08-23T21:19:20.442927', 'updated_at': '2026-08-23T21:19:20.442934'}], 'relations': [{'entity_id': 'index-main', 'relation_type': 'uses_index', 'confidence': 0.9, 'metadata': {}, 'created_at': '2026-08-23T21:19:20.442350', 'updated_at': '2026-08-23T21:19:20.442350'}], 'created_at': '2026-08-23T21:19:20.442350', 'updated_at': '2026-08-23T21:19:20.4

## 6 · `KNOWLEDGE_BASE`

**Unit shape:** source content chunk, namespace, shared
`knowledge_base_id`, chunk index/count, chunking strategy, embedding,
and tenant scope.

**Operations:** `ingest_knowledge`, `ingest_file`, `ingest_directory`,
`retrieve_knowledge`, `retrieve_knowledge_by_query`, and
`attach_to_agent`.

**Why it exists:** knowledge memory grounds an answer in source
passages too large or numerous for the permanent prompt. Namespace,
source identity, and chunk metadata make retrieval and citation
auditable.

**Read the output:** three paragraphs become separately retrievable
chunks; the severity question should rank the incident-lead passage;
attachment makes the knowledge-base ID available to the agent.

**Production practice:** fingerprint source content plus chunking and
embedding configuration, batch embedding work, deduplicate by parent
source, and measure retrieval recall independently from reader quality.
Re-ingest or invalidate when the authoritative document changes.

In [7]:
knowledge = KnowledgeBase(provider)
kb_id = knowledge.ingest_knowledge(
    (
        "Incident severity one requires an incident commander and a communications lead.\n\n"
        "Rollback is preferred when a deployment causes sustained error-budget burn.\n\n"
        "The retrieval service SLO is p95 latency below 250 milliseconds."
    ),
    namespace="operations-handbook",
    chunking_strategy="paragraph",
    user_id=USER_ID,
)
chunks = knowledge.retrieve_knowledge(kb_id)
hits = provider.retrieve_by_query(
    "Who leads a severity one incident?",
    memory_store_type=MemoryType.KNOWLEDGE_BASE,
    namespace="operations-handbook",
    user_id=USER_ID,
    limit=2,
)
knowledge.attach_to_agent(episodic_agent, kb_id)
print({"chunks": len(chunks), "top_hit": hits[0]["content"], "attached": kb_id in episodic_agent.knowledge_base_ids})

{'chunks': 3, 'top_hit': 'Incident severity one requires an incident commander and a communications lead.', 'attached': True}


---
# Part III · Procedural memory

## 7 · `TOOLBOX`

**Unit shape:** tool name, description, strict JSON Schema, policy,
aliases/deprecated arguments, optional semantic metadata, and a trusted
in-process callable binding.

**Security boundary:** executable Python is never reconstructed by
unpickling. Callables must be explicitly rebound or resolved from a
trusted import reference after restart.

**Why it exists:** the model needs a bounded, machine-checkable action
surface. Type hints and docstrings become a strict schema; governance
metadata declares determinism, side effects, and invalidation domains.
A semantic router can disclose only the most relevant allowlisted tools
instead of paying to send every schema on every turn.

**Read the output:** the schema should reject undeclared properties,
the trusted binding executes locally, and registration does not create
an LLM because `augment=False`. Persisted metadata alone is not executable
authority after restart.

**Production boundary:** mutations require durable host approval, not a
model-visible Boolean. Validate arguments again immediately before
dispatch and never reconstruct callable code from database bytes.

In [8]:
@governed_tool(deterministic=True, side_effects=False, domains=("health",))
def service_health(service: str) -> dict:
    '''Read current health plus a bounded diagnostic report.'''
    return {
        "service": service,
        "status": "healthy",
        "diagnostics": [
            f"probe-{index:02d}: healthy" for index in range(24)
        ],
    }


toolbox = Toolbox.from_functions(
    [service_health],
    memory_provider=provider,
    agent_id=episodic_agent.agent_id,
    user_id=USER_ID,
    augment=False,
)
metadata = toolbox.get_tool_by_name("service_health")
bound = toolbox.get_function_by_name("service_health")
bound_result = bound("retrieval-api")
print("Schema:", metadata["input_schema"])
print(
    "Bound call:",
    {
        "service": bound_result["service"],
        "status": bound_result["status"],
        "serialized_chars": len(str(bound_result)),
    },
)
print("Available tools:", [item["name"] for item in toolbox.list_available_tools()])

Schema: {'type': 'object', 'properties': {'service': {'type': 'string', 'description': 'Parameter service'}}, 'additionalProperties': False, 'required': ['service']}
Bound call: {'service': 'retrieval-api', 'status': 'healthy', 'serialized_chars': 570}
Available tools: ['service_health']


## 8 · `WORKFLOW_MEMORY`

**Unit shape:** named step map, user query, outcome, exact scope,
canonical signature/hash, activated skills, shadow evaluations, and
optional promoted-skill pointer.

**Operations:** construct, `add_step`, `store_workflow`, retrieve by
intent, and serialize with `to_dict`/`from_dict`. Runtime tool loops can
capture workflows automatically.

**Why it exists:** workflow memory represents a trajectory—what steps
were taken, with which arguments, dependencies, outcome, and evidence.
Its canonical signature groups materially equivalent procedures so
repeated success or failure can be analyzed.

**Read the output:** a durable row ID proves persistence; the canonical
hash/signature survive serialization. The example is authored for
clarity, while workflow application mode can capture actual tool loops.

**Do not confuse with a skill:** a workflow is evidence of a sequence.
It should not become instruction merely because it occurred once. Verify
outcomes before reuse and never replay stale credentials or approvals.

In [9]:
workflow = Workflow(
    name="incident-triage",
    description="Inspect service health and classify the incident.",
    memory_id=MEMORY_ID,
    agent_id=episodic_agent.agent_id,
    user_id=USER_ID,
    user_query="Triage retrieval-api",
    outcome=WorkflowOutcome.SUCCESS,
)
workflow.add_step("health", {"tool": "service_health", "arguments": {"service": "retrieval-api"}})
workflow.add_step("classify", {"tool": "classify_incident", "arguments": {"policy": "severity"}})
workflow_id = workflow.store_workflow(provider)
matches = Workflow.retrieve_workflows_by_query("triage a service incident", provider, limit=3)
round_trip = Workflow.from_dict(matches[0].to_dict())
print(
    {
        "row_id": workflow_id,
        "canonical_hash": round_trip.canonical_hash,
        "canonical_signature": round_trip.canonical_signature,
    }
)

{'row_id': 'ee856eb0-7f8c-4500-997f-18c0d6c50034', 'canonical_hash': 'b132b7135a834a55cfe76d6617ada1b8987e938b24ecec7cf86977c3c62860a0', 'canonical_signature': [{'tool': 'health', 'arg_keys': ['service'], 'errored': False, 'retry_count': 1}, {'tool': 'classify', 'arg_keys': ['policy'], 'errored': False, 'retry_count': 1}]}


## 9 · `SKILLBOX`

**Unit shape:** name, applicability description, instruction content,
preconditions, tools, example queries, source workflow class, version,
injection authority, lifecycle status, baseline, and outcome stats.

**Lifecycle:** candidate → shadow → active → demoted/deprecated. A skill
is instruction, so false-positive retrieval is more dangerous than an
ordinary document miss.

**Why it exists:** a skill is a reusable strategy or playbook retrieved
only when applicable. It can be authored directly or proposed from a
class of successful workflows. Unlike a knowledge passage, it is meant
to shape behavior.

**Read the output:** semantic retrieval finds the incident-triage skill;
the next calls demote it and verify the lifecycle state. This proves
that retrieval authority is reversible.

**Continual-learning rule:** because LLM instructions have higher
behavioral impact than ordinary evidence, promotion needs verified
outcomes, minimum support, shadow evaluation, versioning, and rollback.
Retrieval must filter by agent/user/status before top-k similarity.

In [10]:
skillbox = Skillbox(provider, agent_id=episodic_agent.agent_id)
skill = Skill(
    name="operations/incident-triage",
    description="Triage an unhealthy production service.",
    content="Check health, classify severity, appoint owners, communicate, then verify recovery.",
    preconditions=["a production service is degraded"],
    tools_used=["service_health"],
    queries=["triage an incident", "service is unhealthy"],
    agent_id=episodic_agent.agent_id,
    user_id=USER_ID,
    source_canonical_hash=workflow.canonical_hash,
    source_workflow_ids=[workflow.workflow_id],
    status=SkillStatus.ACTIVE,
)
skill_row_id = skillbox.add_skill(skill)
skill_hits = skillbox.retrieve_skills_by_query(
    "The production service is unhealthy",
    limit=2,
    min_similarity=0.0,
    user_id=USER_ID,
)
print({"row_id": skill_row_id, "hits": [(hit.skill.name, hit.similarity) for hit in skill_hits]})
print("Demoted:", skillbox.set_status(skill.skill_id, SkillStatus.DEMOTED, reason="Procedure superseded"))
print("Lifecycle status:", skillbox.get_skill_by_id(skill.skill_id).status.value)

{'row_id': 'f814a598-1c7e-40c8-a811-8ca411e6d113', 'hits': [('operations/incident-triage', 0.73333340883255)]}
Demoted: True
Lifecycle status: demoted


## 10 · `TOOL_LOG`

**Unit shape:** tool/call IDs, arguments, result or offloaded full
result, digest, success/error state, timestamp, and exact run scope.

**Owner:** the tool loop. Read with `list_tool_logs` or retrieve one
expansion by ID. Small results stay inline; large results can become an
auditable pointer, and expansion tools are never re-offloaded.

**Why it exists:** large tool payloads can dominate the context window,
while operators still need the complete result for replay and audit.
Size-aware offloading stores the payload once and gives the model a
compact digest plus durable identifier.

**Read the output:** `service_health` intentionally exceeds the
256-character threshold. The response completes normally and the
scoped log list contains one successful execution. A small result would
remain inline and correctly create no row.

**Security:** redact secrets before persistence, retain argument hashes
and approver/trace identity where relevant, and return a structured
expansion error for missing or unauthorized pointers. Expansion tools
are exempt from offloading to prevent pointer-to-pointer loops.

In [11]:
tool_agent = (
    MemAgentBuilder()
    .with_name("Tool Log Guide")
    .with_instruction("Use service_health for current service state.")
    .with_model(TutorialModel())
    .with_memory_provider(provider)
    .with_memory_ids(MEMORY_ID)
    .with_tools([service_health])
    .with_toolbox(toolbox)
    .with_tool_result_policy(ToolResultPolicy(offload_above_chars=256))
    .with_automations_enabled(False)
    .build()
)
print(tool_agent.run("Check service health.", **SCOPE))
logs = tool_agent.memory_manager.list_tool_logs(
    MEMORY_ID, user_id=USER_ID, thread_id=THREAD_ID
)
print(
    [
        {key: row.get(key) for key in ("tool_log_id", "tool_name", "success", "thread_id", "user_id")}
        for row in logs
    ]
)
assert any(row.get("tool_name") == "service_health" for row in logs)

The service is healthy.
[{'tool_log_id': '64c6738d-df1c-4d20-b545-2d2717c361a4', 'tool_name': 'service_health', 'success': True, 'thread_id': 'memory-taxonomy', 'user_id': 'developer-42'}]


---
# Part IV · Working memory

## 11 · `SHORT_TERM_MEMORY`

This is runtime-managed active state, not a public mutable transcript.
MemoRizz assembles a bounded context from the request, recent history,
retrieved long-term memory, summaries, tools, and ephemeral host
context. Inspect it with `get_context_window_stats()` and tune it with
`ContextPolicy` rather than writing arbitrary short-term rows.

**Why it exists:** the model can reason only over the current context
window. Working memory is therefore a per-turn selection problem, not a
promise to persist everything. The host can add current page, selection,
tenant-safe UI state, or task constraints without turning them into
durable facts.

**Read the output:** context statistics show the last prompt/completion
budget and assembly stage. The deployment name is deliberately
ephemeral; promote it only if an authoritative event says it should
survive the turn.

**Failure modes:** prompt overflow, duplicated evidence, stale page
context, or private context attached to the wrong user/thread. Bound
each source and retain provenance for every injected item.

In [12]:
episodic_agent.run(
    "Use the current deployment name only for this turn.",
    **SCOPE,
    context={
        "current_page": {"type": "deployment", "id": "deploy-17", "title": "Retrieval API"},
        "deployment_name": "blue-canary",
    },
)
print(episodic_agent.get_context_window_stats())

{'timestamp': '2026-08-23T22:19:20.486279', 'prompt_tokens': 4156, 'completion_tokens': 12, 'total_tokens': 4168, 'context_window_tokens': None, 'percentage_used': None, 'stage': 'iteration_1'}


## 12 · `SEMANTIC_CACHE`

**Unit shape:** query/response embedding, cache key, model/prompt/tool
and data-version fingerprints, user/session scope, TTL, domains/tags,
hit count, and provenance.

**Operations:** configure, inspect without exposing the response,
monitor counters, invalidate by domain/tag/version, and clear by scope.
Similarity alone never establishes freshness.

**Why it exists:** safe reuse avoids repeating embedding/retrieval/model
work and can reduce latency and hosted-model cost. Cache identity must
include everything that can change the answer—not just a query vector.

**Read the output:** two identical requests produce one model call. The
stats distinguish hit, miss, write, and size; inspection shows the
matched key, score, TTL, age, hit count, and domain. Domain invalidation
then makes the stale answer unavailable.

**Admission policy:** default to read-only deterministic responses.
Bypass side effects, approvals, browser control, time-sensitive state,
or any request whose data version and freshness cannot be represented.

In [13]:
cache_model = TutorialModel()
cache_agent = (
    MemAgentBuilder()
    .with_name("Cache Guide")
    .with_instruction("Answer deterministic read-only questions.")
    .with_model(cache_model)
    .with_memory_provider(provider)
    .with_memory_ids("cache-guide")
    .with_semantic_cache(enabled=True, threshold=0.95, scope="session")
    .with_automations_enabled(False)
    .build()
)
cache_context = {"cache_domains": ["handbook"], "data_version": "v1"}
cache_scope = {
    "memory_id": "cache-guide",
    "user_id": USER_ID,
    "thread_id": "cache-thread",
    "context": cache_context,
}
cache_agent.run("What do I prefer?", **cache_scope)
cache_agent.run("What do I prefer?", **cache_scope)
inspection = cache_agent.inspect_semantic_cache(
    "What do I prefer?",
    user_id=USER_ID,
    thread_id="cache-thread",
    context=cache_context,
)
print({"model_calls": cache_model.calls, "stats": cache_agent.semantic_cache_stats()})
print("Inspection:", inspection.to_dict())
print("Invalidated:", cache_agent.invalidate_semantic_cache(domains=["handbook"]))
assert cache_model.calls == 1 and inspection.hit

{'model_calls': 1, 'stats': {'enabled': True, 'hits': 1, 'misses': 1, 'bypasses': 0, 'writes': 1, 'evictions': 0, 'size': 1, 'bypass_reasons': {}, 'last_hit': {'cache_key': '7c690731-8bcf-5072-9510-b92b7404ed16', 'query': 'What do I prefer?', 'similarity': 1.0, 'age_seconds': 0.0018639564514160156, 'agent_id': 'bdab16e8-cded-5c73-9e27-6b6d24199b12', 'memory_id': 'cache-guide', 'session_id': 'cache-thread', 'user_id': 'developer-42', 'metadata': {'fingerprints': {'model': '99555bf9c846d0b8aab5dd8a24189ff60cf89e09cdb82c4adc99cc5a99cadb09', 'prompt': 'a8feaa7acbd5451eac5d59dade52ba2818e70a23b382304ef46e47225b37a519', 'tool_schema': 'c8177d5db0a1484f7e2868070e486b7d1c0c4f0ab5d86f7263c8bb4540bc13eb', 'completion_policy': 'e0b81db2190bb1955538db6433846a0880923ce028738c1951fd21fc4edad1b0', 'data_version': 'v1', 'request_context': '2d24ea05321e36fef0aa0ae2054ca84b9b41e9df9c1e503a6a412f59a69c0a59'}, 'domain': 'handbook', 'domains': ['handbook'], 'tags': [], 'admission': {'deterministic': True, 

---
# Part V · Social and durable memory

## 13 · `SHARED_MEMORY`

**Unit shape:** an owned session with participants, workflow/user/trace
scope, status, and a blackboard of typed command, status, and report
messages.

**Operations:** create a session, post typed messages, filter entries,
update status, register sub-agents, and inspect hierarchy.

**Why it exists:** collaboration needs more structure than concatenating
every agent's transcript. Commands define responsibility; status events
expose progress and dependency state; reports carry findings and
citations back to the coordinator.

**Read the output:** the blackboard preserves command → status → report
order and the identity of each writer. In production, propagate the
workflow, tenant, trace, and request context into every participant.

**Do not use:** a process-global scratchpad shared across unrelated
workflows. Apply workflow-scoped retention and expose partial failures
rather than presenting a partial team result as complete.

In [14]:
shared = SharedMemory(provider)
shared_id = shared.create_shared_session(
    root_agent_id="lead",
    delegate_agent_ids=["analyst"],
    workflow_id="memory-review",
    user_id=USER_ID,
    trace_id="trace-memory-review",
)
shared.post_command(
    shared_id,
    agent_id="lead",
    command_id="review-1",
    target_agent_id="analyst",
    instructions="Audit the memory taxonomy.",
)
shared.post_status(
    shared_id,
    agent_id="analyst",
    command_id="review-1",
    status="in_progress",
    progress=50,
)
shared.post_report(
    shared_id,
    agent_id="analyst",
    command_id="review-1",
    findings="Runtime-owned and application-authored units are clearly separated.",
)
shared.update_session_status(shared_id, "completed")
print(shared.get_blackboard_entries(shared_id))

[{'memory_id': 'f71f9163-fde8-4332-b31f-88c8aca739b1', 'agent_id': 'lead', 'content': {'message_id': '30addaa0-a897-47ac-98d4-91ff23511a6a', 'message_type': 'COMMAND', 'created_at': '2026-08-23T21:19:20.504497', 'payload': {'command_id': 'review-1', 'target_agent_id': 'analyst', 'instructions': 'Audit the memory taxonomy.', 'priority': 3, 'dependencies': [], 'metadata': {}}}, 'entry_type': 'COMMAND', 'created_at': '2026-08-23T22:19:20.520287'}, {'memory_id': '93fcb56b-ae18-4474-a300-7fd5754f62dd', 'agent_id': 'analyst', 'content': {'message_id': '6e7563cd-1a77-4b93-b13a-853fd6b5c5af', 'message_type': 'STATUS', 'created_at': '2026-08-23T21:19:20.520915', 'payload': {'command_id': 'review-1', 'agent_id': 'analyst', 'status': 'in_progress', 'progress': 50, 'blockers': None, 'summary_ids': []}}, 'entry_type': 'STATUS', 'created_at': '2026-08-23T22:19:20.520978'}, {'memory_id': 'eef199de-9784-4ebb-9cd1-fa9f94ffea4c', 'agent_id': 'analyst', 'content': {'message_id': 'fec81bee-fe71-4a7f-92bd-

## 14 · `MEMAGENT`

**Unit shape:** agent identity, instruction, provider/model references,
active memory IDs/types, persona snapshot, tool and policy metadata,
application mode, and capability configuration. Secrets and executable
code are not serialized.

**Operations:** `build_and_save`, `save`, `MemAgent.load`, `refresh`,
`list_memagents`, and `delete_memagent(..., cascade=True)`.

**Why it exists:** a memory-first agent is configuration plus durable
references, not only a live Python object. Persisting the definition
lets another process reconstruct the same instruction, memory links,
application mode, and policies.

**Read the output:** the stored and restored identities match and the
agent appears in the provider listing. The model object is supplied by
the trusted host during load because credentials, clients, and
executable code are intentionally not serialized.

**Lifecycle:** save reviewed configuration changes, refresh long-lived
processes when desired, and use cascade deletion only with exact scope
and an explicit retention decision.

In [15]:
stored_agent = provider.retrieve_memagent(episodic_agent.agent_id)
restored_agent = MemAgent.load(
    episodic_agent.agent_id,
    memory_provider=provider,
    model=TutorialModel(),
)
print(
    {
        "stored_agent_id": stored_agent.agent_id,
        "memory_ids": stored_agent.memory_ids,
        "restored_name": restored_agent.name,
        "agent_count": len(provider.list_memagents()),
    }
)

{'stored_agent_id': '6171481f-8ccf-5547-9d7b-b6305bf6a010', 'memory_ids': ['memory-types-demo'], 'restored_name': 'Memory Types Guide', 'agent_count': 3}


## Complete operation matrix

| Store | Normal writer | Read/search | Update/lifecycle | Delete/retention |
|---|---|---|---|---|
| `CONVERSATION_MEMORY` | `agent.run` | scoped history | summary marker via compaction | provider retention/agent cascade |
| `SUMMARIES` | `generate_summaries` | `fetch_context_summary` | regenerate/version policy | provider retention |
| `PERSONAS` | `store_persona`, `set_persona` | retrieve/list/similar | `Persona.update` | `delete_persona` |
| `ENTITY_MEMORY` | upsert/record attribute | get/list/search diagnostics | merge with confidence/source | provider delete by stored row ID |
| `KNOWLEDGE_BASE` | ingest text/file/directory | by ID/query/namespace | re-ingest or provider admin update | provider delete by chunk IDs |
| `TOOLBOX` | `from_functions`, register | list/get/similar | bind/update trusted metadata | delete by name/ID/all |
| `WORKFLOW_MEMORY` | runtime capture or `store_workflow` | intent retrieval | outcome/promotion metadata | provider retention |
| `SKILLBOX` | authored skill or promotion | scoped active/shadow retrieval | status, version, monitor, demotion | deprecate or provider retention |
| `TOOL_LOG` | tool runtime | list/retrieve/expand | immutable audit record | scoped retention |
| `SHORT_TERM_MEMORY` | context runtime | context telemetry | context policy | expires/runtime cleanup |
| `SEMANTIC_CACHE` | admitted read-only result | inspect/stats | TTL and domain/version invalidation | clear/invalidate |
| `SHARED_MEMORY` | orchestrator/coordination | entries/messages/hierarchy | status and participants | workflow retention |
| `MEMAGENT` | `save`, `build_and_save` | load/list/refresh | save new configuration | delete, optionally cascade |

## How to choose a memory type

```mermaid
flowchart TD
  Q{What must persist?}
  Q -->|What happened?| E[Conversation memory]
  E -->|History is too large| S[Summaries]
  Q -->|What is true?| T{Shape}
  T -->|Document/passages| K[Knowledge base]
  T -->|Structured object| N[Entity memory]
  T -->|Agent identity| P[Persona]
  Q -->|How to act?| H{Representation}
  H -->|Callable capability| B[Toolbox]
  H -->|Observed trajectory| W[Workflow memory]
  H -->|Reviewed instruction| L[Skillbox]
  Q -->|Reuse an answer?| C[Semantic cache]
  Q -->|Coordinate agents?| M[Shared memory]
```

Use more than one type only when each representation owns a distinct
invariant. Duplicating the same fact across every store makes updates,
forgetting, and provenance harder—not smarter.

A practical selection test:

1. Does chronology matter? Start with conversation memory.
2. Is it a stable structured fact? Use entity memory.
3. Must a source passage support the claim? Use a knowledge base.
4. Is it executable capability, observed sequence, or instruction?
   Choose toolbox, workflow, or skill respectively.
5. Is it only useful for this turn? Keep it in working context.
6. Is it safe reuse of a prior answer? Consider semantic cache.

If no durable future decision improves, do not form a memory unit.

## Provider portability and production checks

The same logical units work on filesystem, MongoDB, and Oracle, but
production readiness also requires:

- exact tenant/thread scoping before top-k retrieval;
- embedding dimension and model compatibility;
- atomic summary creation and source marking;
- durable approvals for side effects;
- cache freshness and invalidation policy;
- retention, deletion, and audit requirements;
- retrieval and answer-quality evaluation on your own workload.

Use `provider.memory_capabilities()`, Oracle `preflight()`,
`agent.capability_report()`, and `agent.observability_summary()` as
operational evidence rather than assuming all backends are identical.

### Design review checklist

For every proposed memory unit, document its schema, authoritative
writer, tenant boundary, retrieval path, ranking signals, source
provenance, freshness rule, update/supersession behavior, retention,
deletion, and evaluation metric. This turns “add memory” from a vague
feature request into an engineering contract.

### Exercises

- Add a conflicting entity attribute and implement a documented source
  precedence rule.
- Change the cache data version and verify a miss without clearing the
  whole provider.
- Expand a stored summary and compare every source message.
- Register a second tool, enable progressive disclosure, and inspect
  which schema is visible for two different queries.
- Repeat the lifecycle with the Oracle notebook and compare provider
  capabilities rather than assuming identical behavior.

In [16]:
print("Provider capabilities:", provider.memory_capabilities().to_dict())
print(
    "Observability:",
    episodic_agent.observability_summary(MEMORY_ID, USER_ID, thread_id=THREAD_ID),
)

Provider capabilities: {'provider': 'FileSystemProvider', 'batch_store': True, 'transactional_batch': False, 'scoped_search': True, 'result_scores': True, 'provenance': True, 'native_vector_search': False, 'native_hybrid_search': False}
Observability: {'agent_id': '6171481f-8ccf-5547-9d7b-b6305bf6a010', 'memory_id': 'memory-types-demo', 'user_id': 'developer-42', 'thread_id': 'memory-taxonomy', 'conversation': {'row_count': 6, 'message_count': 6, 'role_counts': {'user': 3, 'assistant': 3}, 'thread_count': 1, 'summarized_count': 4, 'trace_bundle_count': 0, 'trace_event_count': 0, 'first_timestamp': 1787516360.433663, 'last_timestamp': 1787516360.487896}, 'tool_logs': {'count': 0, 'failure_count': 0, 'success_count': 0}, 'workflows': {'count': 1, 'outcomes': {'success': 1}}, 'summaries': {'count': 1}, 'approvals': {'count': 0, 'statuses': {}}, 'semantic_cache': {'enabled': False, 'hits': 0, 'misses': 0, 'size': 0}, 'context_window': {'timestamp': '2026-08-23T22:19:20.486279', 'prompt_tok

In [17]:
for agent in (restored_agent, episodic_agent, tool_agent, cache_agent):
    agent.close(close_memory_provider=False)
provider.close()
print("Tutorial resources closed; isolated temporary files may now be removed.")

Tutorial resources closed; isolated temporary files may now be removed.
